# 04. Embeddings: naive UDF vs batched pandas_udf

Times the same rows through a row-by-row UDF and a batched pandas_udf and saves the measured numbers.

## What to look at

- The **results table**: rows/sec for each approach and the measured speedup (whatever it turns out to be).
- The **anti-pattern** row is run on a tiny sample only; compare rows/sec.
- `min_cosine_naive_vs_batched` should be ~1.0: both produce the same vectors.

> This step **overwrites** `benchmarks/embedding_benchmark.*` with the numbers from this run. Use `EMBED_BENCH_ROWS` in `.env` to size it.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
print("rows to benchmark:", common.env_int("EMBED_BENCH_ROWS", 20000), "| model:", common.EMBEDDING_MODEL)

In [ ]:
run_script("04_embed_naive_vs_batched.py")

In [ ]:
j = json.loads((BENCH_DIR / "embedding_benchmark.json").read_text())
pd.DataFrame([
    ("Naive udf (1 row per encode)", j["rows"], j["naive_seconds"], j["naive_rows_per_sec"], 1.0),
    (f"Batched pandas_udf (batch_size={j['batch_size']})", j["rows"], j["batched_seconds"], j["batched_rows_per_sec"], j["speedup_batched_vs_naive"]),
    ("ANTI-PATTERN: load model every row", j["antipattern_rows"], j["antipattern_seconds"], j["antipattern_rows_per_sec"], None),
], columns=["approach", "rows", "seconds", "rows/sec", "speedup vs naive"])

In [ ]:
print(f"Batched was {j['speedup_batched_vs_naive']}x the naive UDF (measured in this run).")
print("min cosine(naive, batched):", j["min_cosine_naive_vs_batched"])

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

Nothing new. This step reads Bronze and does its work with the `noop` writer, so no files are written.